# 05 · SOC benchmark and the SOH-shortcut ablation (R1-1)

Needs the raw discharge waveforms of the six SOC cells under
`organized/<cell>/discharge/<file>.csv` (or set the environment variable
`PCOE_ORGANIZED`). Feature sets: A = paper (SOH_k, centered dV/dt);
B = previous-cycle SOH, backward differences (causal); C = no SOH;
D = no SOH and no charge removed. Baselines: OCV lookup, CC-nominal (rated 2 Ah),
CC-prev (previous cycle's capacity).

In [1]:
%run ./00_common.ipynb

root: <repository root>


In [2]:
meta, qrefs = soc_load_meta()
tab = soc_build_table(meta)
ncyc = tab.groupby("battery_id")["cycle"].nunique()
print(f"SOC table: {len(tab)} samples, {ncyc.sum()} cycles", ncyc.to_dict())
q = pd.DataFrame([dict(battery=b, temp=qrefs[b]["temp"], q_ref_Ah=round(qrefs[b]["q_ref"], 4),
                       slope_100_over_qref=round(100 / qrefs[b]["q_ref"], 2), n_soc_cycles=int(ncyc[b]),
                       group="24C-in-group" if b in GROUP24 else "out-of-group") for b in SOC_CELLS])
q.to_csv(RESULTS / "soc_qref.csv", index=False)
q

SOC table: 47144 samples, 164 cycles {'B0005': 34, 'B0006': 34, 'B0007': 34, 'B0018': 27, 'B0042': 14, 'B0054': 21}


,battery,temp,q_ref_Ah,slope_100_over_qref,n_soc_cycles,group
0,B0005,24.0,1.8565,53.87,34,24C-in-group
1,B0006,24.0,2.0353,49.13,34,24C-in-group
2,B0007,24.0,1.8911,52.88,34,24C-in-group
3,B0018,24.0,1.8550,53.91,27,24C-in-group
4,B0042,22.0,1.7315,57.75,14,out-of-group
5,B0054,4.0,1.1665,85.72,21,out-of-group


In [3]:
def met(y, p):
    e = np.asarray(p) - np.asarray(y)
    return dict(RMSE=float(np.sqrt((e ** 2).mean())), MAE=float(np.abs(e).mean()),
                R2=float(1 - (e ** 2).sum() / ((y - y.mean()) ** 2).sum()), MaxErr=float(np.abs(e).max()))


def ocv(train, test):
    from scipy.interpolate import interp1d
    ref = pd.concat([g[g["cycle"] == g["cycle"].min()] for _, g in train.groupby("battery_id")])
    o = ref.groupby(pd.cut(ref["V"], bins=60), observed=True).agg(V=("V", "mean"), soc=("soc", "mean")).dropna()
    f = interp1d(o["V"], o["soc"], bounds_error=False, fill_value=(o["soc"].iloc[0], o["soc"].iloc[-1]))
    return np.clip(f(test["V"].to_numpy()), -5, 105)

In [4]:
t0 = time.time()
rows = []
for held in SOC_CELLS:
    tr, te = tab[tab.battery_id != held], tab[tab.battery_id == held]
    yte = te["soc"].to_numpy()
    rows.append(dict(config="baseline", protocol="LOBO", held_out=held, model="OCV", **met(yte, ocv(tr, te))))
    rows.append(dict(config="baseline", protocol="LOBO", held_out=held, model="CC-nominal",
                     **met(yte, np.clip(100 * (Q_NOMINAL - te["q_removed"]) / Q_NOMINAL, -5, 105))))
    rows.append(dict(config="baseline", protocol="LOBO", held_out=held, model="CC-prev",
                     **met(yte, np.clip(100 * (te["cap_prev"] - te["q_removed"]) / te["qref"], -5, 105))))
    for cfgname, cc in SOC_FEATSETS.items():
        for name, m in soc_models().items():
            m.fit(tr[cc].to_numpy(float), tr["soc"].to_numpy())
            rows.append(dict(config=cfgname, protocol="LOBO", held_out=held, model=name, **met(yte, m.predict(te[cc].to_numpy(float)))))
    print(f"LOBO {held} done ({time.time() - t0:.0f}s)")
for b in SOC_CELLS:
    g = tab[tab.battery_id == b]
    cyc = np.sort(g["cycle"].unique()); split = cyc[int(len(cyc) * 0.6)]
    tr, te = g[g.cycle < split], g[g.cycle >= split]
    for cfgname, cc in SOC_FEATSETS.items():
        for name, m in soc_models().items():
            m.fit(tr[cc].to_numpy(float), tr["soc"].to_numpy())
            rows.append(dict(config=cfgname, protocol="SameCell", held_out=b, model=name,
                             **met(te["soc"].to_numpy(), m.predict(te[cc].to_numpy(float)))))
res = pd.DataFrame(rows)
res["group"] = np.where(res["held_out"].isin(GROUP24), "in24", "out")
res.to_csv(RESULTS / "soc_ablation.csv", index=False)
res.groupby(["config", "protocol", "model"])["RMSE"].mean().unstack("protocol").round(2)

LOBO B0005 done (30s)


LOBO B0006 done (60s)


LOBO B0007 done (88s)


LOBO B0018 done (120s)


LOBO B0042 done (157s)


LOBO B0054 done (192s)


protocol                        LOBO  SameCell
config           model                        
A_paper_SOHk     LightGBM       6.48      6.65
                 Linear         6.94      1.67
                 RandomForest   6.69      7.34
                 Ridge          6.92      2.11
                 XGBoost        6.73      6.53
B_SOHprev_causal LightGBM       6.84      6.69
                 Linear         5.67      3.94
                 RandomForest   7.06      7.38
                 Ridge          5.67      4.13
                 XGBoost        6.88      6.60
C_noSOH          LightGBM       7.45      7.68
                 Linear         9.19     13.14
                 RandomForest   7.67      8.17
                 Ridge          9.17     14.24
                 XGBoost        7.42      7.46
D_noSOH_noQ      LightGBM      11.20      8.44
                 Linear        11.96     40.15
                 RandomForest  11.58      8.49
                 Ridge         11.95     40.47
                 XGBoost       10.31      8.34
baseline         CC-nominal    24.61       NaN
                 CC-prev        2.59       NaN
                 OCV           16.58       NaN

In-group (24 °C) versus out-of-group LOBO RMSE: the gap survives removing SOH.

In [5]:
res[res.protocol == "LOBO"].groupby(["config", "model", "group"])["RMSE"].mean().unstack("group").round(2)

group                           in24    out
config           model                     
A_paper_SOHk     LightGBM       1.58  16.26
                 Linear         2.68  15.46
                 RandomForest   1.83  16.43
                 Ridge          2.66  15.44
                 XGBoost        2.16  15.86
B_SOHprev_causal LightGBM       1.78  16.95
                 Linear         2.94  11.14
                 RandomForest   2.32  16.54
                 Ridge          2.93  11.14
                 XGBoost        2.15  16.33
C_noSOH          LightGBM       2.50  17.35
                 Linear         7.66  12.25
                 RandomForest   2.81  17.40
                 Ridge          7.69  12.12
                 XGBoost        2.54  17.19
D_noSOH_noQ      LightGBM       3.50  26.60
                 Linear         8.81  18.27
                 RandomForest   3.70  27.34
                 Ridge          8.80  18.27
                 XGBoost        3.48  23.99
baseline         CC-nominal    21.21  31.42
                 CC-prev        1.19   5.38
                 OCV           13.21  23.32